In [1]:
import pandas as pd

model = pd.read_csv("../data/processed/model_dataset.csv")

print(model.shape)
print(model.columns.tolist())

(9932614, 47)
['date', 'receipt_id', 'store_id', 'sku_id', 'customer_id', 'quantity', 'unit_price', 'total_value', 'channel', 'discount_pct', 'promo_id', 'year', 'month', 'quarter', 'week', 'day_of_week', 'is_weekend', 'total_quantity', 'total_sales', 'average_price', 'total_transactions', 'sku_name', 'category', 'subcategory', 'unit_price_sku', 'cost_price', 'brand', 'total_spent', 'total_quantity_customer', 'total_transactions_customer', 'average_transaction_value', 'cust_id', 'age', 'gender', 'city', 'loyalty_segment', 'preferred_channel', 'registration_date', 'total_sales_store', 'total_quantity_store', 'total_transactions_store', 'average_order_value', 'store_name', 'city_store', 'store_type', 'opening_date', 'promotion_used']


In [2]:
for i, col in enumerate(model.columns):
    print(i, col)

0 date
1 receipt_id
2 store_id
3 sku_id
4 customer_id
5 quantity
6 unit_price
7 total_value
8 channel
9 discount_pct
10 promo_id
11 year
12 month
13 quarter
14 week
15 day_of_week
16 is_weekend
17 total_quantity
18 total_sales
19 average_price
20 total_transactions
21 sku_name
22 category
23 subcategory
24 unit_price_sku
25 cost_price
26 brand
27 total_spent
28 total_quantity_customer
29 total_transactions_customer
30 average_transaction_value
31 cust_id
32 age
33 gender
34 city
35 loyalty_segment
36 preferred_channel
37 registration_date
38 total_sales_store
39 total_quantity_store
40 total_transactions_store
41 average_order_value
42 store_name
43 city_store
44 store_type
45 opening_date
46 promotion_used


In [3]:
import os

files = os.listdir("../data/processed")

for f in files:
    print(f)

.gitkeep
customer_master_clean.csv
inventory_snapshot_clean.csv
model_dataset.csv
promotions_clean.csv
risk_scoring_output.csv
sales_transactions_clean.csv
sku_inventory_flags_clean.csv
sku_master_clean.csv
store_master_clean.csv


In [4]:
inventory = pd.read_csv("../data/processed/inventory_snapshot_clean.csv")
sales = pd.read_csv("../data/processed/sales_transactions_clean.csv")
risk = pd.read_csv("../data/processed/risk_scoring_output.csv")

print("INVENTORY")
print(inventory.columns.tolist())

print("\nSALES")
print(sales.columns.tolist())

print("\nRISK")
print(risk.columns.tolist())

INVENTORY
['store_id', 'sku_id', 'stock_on_hand', 'reorder_point', 'safety_stock', 'last_restock_date']

SALES
['date', 'receipt_id', 'store_id', 'sku_id', 'customer_id', 'quantity', 'unit_price', 'total_value', 'channel', 'discount_pct', 'promo_id']

RISK
['store_id', 'sku_id', 'stock_on_hand', 'reorder_point', 'safety_stock', 'last_restock_date', 'total_quantity', 'average_weekly_demand', 'total_sales', 'average_unit_price', 'cost_price', 'inventory_coverage_weeks', 'stockout_risk', 'overstock_risk', 'risk_score', 'risk_level', 'recommended_action', 'inventory_value', 'stockout_quantity_exposure', 'stockout_value_exposure', 'excess_inventory_quantity', 'excess_inventory_value']


In [5]:
import pandas as pd
import numpy as np

# Load datasets
sales = pd.read_csv("../data/processed/sales_transactions_clean.csv")
inventory = pd.read_csv("../data/processed/inventory_snapshot_clean.csv")
risk = pd.read_csv("../data/processed/risk_scoring_output.csv")
sku = pd.read_csv("../data/processed/sku_master_clean.csv")

# Make sure date is datetime
sales["date"] = pd.to_datetime(sales["date"])

# Monthly actual demand by SKU
monthly_sales = (
    sales.groupby(
        [sales["date"].dt.to_period("M"), "sku_id"],
        as_index=False
    )["quantity"]
    .sum()
)

monthly_sales["date"] = monthly_sales["date"].dt.to_timestamp()

# Calculate 3-month rolling forecast
monthly_sales = monthly_sales.sort_values(["sku_id", "date"])

monthly_sales["forecast"] = (
    monthly_sales
    .groupby("sku_id")["quantity"]
    .transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
)

# Rename actual demand
monthly_sales = monthly_sales.rename(columns={
    "quantity": "actual"
})

# Get SKU category
sku_info = sku[["sku_id", "category"]].drop_duplicates()

# Add category
planning = monthly_sales.merge(
    sku_info,
    on="sku_id",
    how="left"
)

# Add inventory information
planning = planning.merge(
    inventory[
        ["sku_id", "stock_on_hand", "reorder_point"]
    ].drop_duplicates("sku_id"),
    on="sku_id",
    how="left"
)

# Add risk information
planning = planning.merge(
    risk[
        [
            "sku_id",
            "risk_level",
            "recommended_action",
            "risk_score"
        ]
    ].drop_duplicates("sku_id"),
    on="sku_id",
    how="left"
)

# Rename columns for dashboard
planning = planning.rename(columns={
    "sku_id": "sku",
    "stock_on_hand": "stock",
    "risk_level": "risk",
    "recommended_action": "action",
    "risk_score": "priority"
})

# Remove rows where forecast cannot be calculated
planning = planning.dropna(subset=["forecast"])

# Round numeric values
planning["forecast"] = planning["forecast"].round(2)
planning["actual"] = planning["actual"].round(2)

# Save
planning.to_csv(
    "../data/processed/planning_dataset.csv",
    index=False
)

print("Planning dataset created!")
print("Shape:", planning.shape)
print("Columns:")
print(planning.columns.tolist())
print(planning.head())

Planning dataset created!
Shape: (235000, 10)
Columns:
['date', 'sku', 'actual', 'forecast', 'category', 'stock', 'reorder_point', 'risk', 'action', 'priority']
        date       sku  actual  forecast        category  stock  \
1 2022-02-01  SKU00001      86     74.00  Home & Kitchen  215.0   
2 2022-03-01  SKU00001      67     80.00  Home & Kitchen  215.0   
3 2022-04-01  SKU00001     103     75.67  Home & Kitchen  215.0   
4 2022-05-01  SKU00001      59     85.33  Home & Kitchen  215.0   
5 2022-06-01  SKU00001      64     76.33  Home & Kitchen  215.0   

   reorder_point  risk                action  priority  
1          100.0  High  Reduce replenishment       3.0  
2          100.0  High  Reduce replenishment       3.0  
3          100.0  High  Reduce replenishment       3.0  
4          100.0  High  Reduce replenishment       3.0  
5          100.0  High  Reduce replenishment       3.0  


In [6]:
planning = pd.read_csv("../data/processed/planning_dataset.csv")

print(planning.shape)
print(planning.isna().sum())
print(planning["sku"].nunique())
print(planning["category"].nunique())
print(planning["risk"].value_counts())
print(planning["action"].value_counts())

(235000, 10)
date                 0
sku                  0
actual               0
forecast             0
category             0
stock            23735
reorder_point    23735
risk             23735
action           23735
priority         23735
dtype: int64
5000
12
risk
High      173383
Medium     37600
Low          282
Name: count, dtype: int64
action
Reduce replenishment     131506
Review replenishment      38164
Reorder urgently          30174
Monitor inventory         11139
Maintain normal stock       282
Name: count, dtype: int64


In [7]:
planning[planning["stock"].isna()][
    ["date", "sku", "actual", "forecast", "category"]
].head(20)

,date,sku,actual,forecast,category
329,2022-02-01,SKU00008,20,26.00,Frozen Foods
330,2022-03-01,SKU00008,26,23.00,Frozen Foods
331,2022-04-01,SKU00008,12,24.00,Frozen Foods
332,2022-05-01,SKU00008,32,19.33,Frozen Foods
333,2022-06-01,SKU00008,37,23.33,Frozen Foods
334,2022-07-01,SKU00008,21,27.00,Frozen Foods
335,2022-08-01,SKU00008,33,30.00,Frozen Foods
336,2022-09-01,SKU00008,36,30.33,Frozen Foods
337,2022-10-01,SKU00008,35,30.00,Frozen Foods
338,2022-11-01,SKU00008,37,34.67,Frozen Foods


In [8]:
print(planning[planning["stock"].isna()]["sku"].nunique())
print(planning[planning["stock"].isna()]["category"].value_counts())

505
category
Dairy & Bakery               2444
Beverages                    2115
Home & Kitchen               2115
Home Care                    2068
Stationery & Office          2021
Frozen Foods                 1974
Electronics & Accessories    1974
Grocery                      1927
Health & Wellness            1880
Snacks & Confectionery       1880
Personal Care                1739
Apparel & Footwear           1598
Name: count, dtype: int64


In [9]:
inventory = pd.read_csv("../data/processed/inventory_snapshot_clean.csv")

print(inventory.shape)
print(inventory.columns.tolist())
print(inventory["sku_id"].nunique())

(26408, 6)
['store_id', 'sku_id', 'stock_on_hand', 'reorder_point', 'safety_stock', 'last_restock_date']
4495


In [10]:
missing_skus = planning.loc[
    planning["stock"].isna(), "sku"
].unique()

print("Missing SKUs:", len(missing_skus))
print(missing_skus[:20])

Missing SKUs: 505
<ArrowStringArray>
['SKU00008', 'SKU00010', 'SKU00022', 'SKU00029', 'SKU00042', 'SKU00054',
 'SKU00055', 'SKU00072', 'SKU00088', 'SKU00089', 'SKU00101', 'SKU00109',
 'SKU00121', 'SKU00132', 'SKU00139', 'SKU00148', 'SKU00152', 'SKU00158',
 'SKU00164', 'SKU00165']
Length: 20, dtype: str


In [11]:
print("Planning SKUs:", planning["sku"].nunique())
print("Inventory SKUs:", inventory["sku_id"].nunique())

inventory_skus = set(inventory["sku_id"])

missing = sorted(set(planning["sku"]) - inventory_skus)

print("SKUs missing from inventory:", len(missing))
print(missing[:20])

Planning SKUs: 5000
Inventory SKUs: 4495
SKUs missing from inventory: 505
['SKU00008', 'SKU00010', 'SKU00022', 'SKU00029', 'SKU00042', 'SKU00054', 'SKU00055', 'SKU00072', 'SKU00088', 'SKU00089', 'SKU00101', 'SKU00109', 'SKU00121', 'SKU00132', 'SKU00139', 'SKU00148', 'SKU00152', 'SKU00158', 'SKU00164', 'SKU00165']


In [12]:
risk = pd.read_csv("../data/processed/risk_scoring_output.csv")

print("Risk SKUs:", risk["sku_id"].nunique())

risk_skus = set(risk["sku_id"])

missing_from_risk = sorted(set(planning["sku"]) - risk_skus)

print("SKUs missing from risk:", len(missing_from_risk))
print(missing_from_risk[:20])

Risk SKUs: 4495
SKUs missing from risk: 505
['SKU00008', 'SKU00010', 'SKU00022', 'SKU00029', 'SKU00042', 'SKU00054', 'SKU00055', 'SKU00072', 'SKU00088', 'SKU00089', 'SKU00101', 'SKU00109', 'SKU00121', 'SKU00132', 'SKU00139', 'SKU00148', 'SKU00152', 'SKU00158', 'SKU00164', 'SKU00165']


In [13]:
valid_skus = set(inventory["sku_id"]) & set(risk["sku_id"])

planning_clean = planning[
    planning["sku"].isin(valid_skus)
].copy()

print("Clean planning shape:", planning_clean.shape)
print("SKUs:", planning_clean["sku"].nunique())
print("Missing values:")
print(planning_clean.isna().sum())

Clean planning shape: (211265, 10)
SKUs: 4495
Missing values:
date             0
sku              0
actual           0
forecast         0
category         0
stock            0
reorder_point    0
risk             0
action           0
priority         0
dtype: int64


In [14]:
print("Duplicate rows:", planning_clean.duplicated().sum())

print("\nRisk distribution:")
print(planning_clean["risk"].value_counts())

print("\nAction distribution:")
print(planning_clean["action"].value_counts())

print("\nPriority distribution:")
print(planning_clean["priority"].value_counts())

Duplicate rows: 0

Risk distribution:
risk
High      173383
Medium     37600
Low          282
Name: count, dtype: int64

Action distribution:
action
Reduce replenishment     131506
Review replenishment      38164
Reorder urgently          30174
Monitor inventory         11139
Maintain normal stock       282
Name: count, dtype: int64

Priority distribution:
priority
3.0    173383
2.0     37600
1.0       282
Name: count, dtype: int64


In [15]:
planning_clean.to_csv(
    "../data/processed/planning_dataset_final.csv",
    index=False
)

print("Final planning dataset saved!")

Final planning dataset saved!
